# Pocket OTC AI Image Analyzer — Google Colab
تشغيل واجهة الويب + Telegram Bot معًا.

**READ-ONLY:** لا يسجل الدخول إلى Pocket Option ولا ينفذ صفقات.

In [ ]:
!rm -rf /content/Jjjjjjj
!git clone -q https://github.com/mohmb142/Jjjjjjj.git /content/Jjjjjjj
%cd /content/Jjjjjjj
!python -m pip install -q -r colab_requirements.txt
print('✅ تم تثبيت المشروع')

In [ ]:
import os
from getpass import getpass
telegram_token = getpass('🔐 Telegram Bot Token: ').strip()
openrouter_key = getpass('🔐 OpenRouter API Key: ').strip()
if not telegram_token or not openrouter_key:
    raise ValueError('يجب إدخال Telegram Bot Token و OpenRouter API Key')
os.environ['TELEGRAM_BOT_TOKEN'] = telegram_token
os.environ['OPENROUTER_API_KEY'] = openrouter_key
os.environ['OPENROUTER_MODEL'] = 'google/gemini-2.5-flash'
print('✅ تم إعداد المفاتيح في جلسة Colab فقط')

In [ ]:
import threading, time, socket
import uvicorn
from google.colab.output import eval_js
from IPython.display import display, HTML

PORT = 8000

def port_ready(host='127.0.0.1', port=PORT):
    for _ in range(40):
        try:
            with socket.create_connection((host, port), timeout=0.5):
                return True
        except OSError:
            time.sleep(0.25)
    return False

def start_web():
    uvicorn.run('main:app', host='0.0.0.0', port=PORT, log_level='info')

web_thread = threading.Thread(target=start_web, daemon=True, name='fastapi-web')
web_thread.start()

if not port_ready():
    raise RuntimeError('لم تبدأ واجهة FastAPI على المنفذ 8000')

public_url = eval_js(f"google.colab.kernel.proxyPort({PORT})")
print('======================================')
print('🌐 Pocket OTC AI Analyzer Web UI')
print('======================================')
print(public_url)
display(HTML(f'''<div style="padding:16px;font-family:Arial">
<h2>🌐 Pocket OTC AI Analyzer</h2>
<a href="{public_url}" target="_blank" style="display:inline-block;padding:12px 18px;background:#1976d2;color:white;text-decoration:none;border-radius:8px">🚀 فتح الواجهة</a>
</div>'''))


In [ ]:
import threading
from telegram_bot import run

def start_telegram():
    print('🤖 Telegram Bot بدأ العمل...')
    run()

telegram_thread = threading.Thread(target=start_telegram, daemon=True, name='telegram-bot')
telegram_thread.start()
print('✅ Telegram Bot يعمل في الخلفية')
print('✅ Web UI تعمل في الخلفية')
print('🌐 الرابط:', public_url)

In [ ]:
import requests
r = requests.get('http://127.0.0.1:8000/', timeout=10)
print('HTTP status:', r.status_code)
if r.status_code == 200:
    print('✅ اختبار واجهة الويب نجح')
else:
    print('⚠️ الواجهة تعمل لكن أعادت حالة:', r.status_code)